## Homework #6
###  Question 1

In [1]:
import pyspark
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/03/12 20:43:37 WARN Utils: Your hostname, codespaces-d3f079 resolves to a loopback address: 127.0.0.1; using 10.0.1.92 instead (on interface eth0)
26/03/12 20:43:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 20:43:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/12 20:43:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
spark.version

'3.5.1'

###  Question 2

In [3]:
input_path = "data/raw/yellow/2025/11/yellow_tripdata_2025_11.parquet"
df_yellow = spark.read.parquet(input_path)

df_yellow_repartitioned = df_yellow.repartition(4)

output_path = "data/pq/yellow/2025/11/"
df_yellow_repartitioned.write.parquet(output_path, mode="overwrite")

In [4]:
!ls -lh data/pq/yellow/2025/11/

total 102M
-rw-r--r-- 1 codespace codespace   0 Mar 12 20:52 _SUCCESS
-rw-r--r-- 1 codespace codespace 26M Mar 12 20:52 part-00000-646db333-5c2c-4aa6-8d0c-fdaa759108df-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar 12 20:52 part-00001-646db333-5c2c-4aa6-8d0c-fdaa759108df-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar 12 20:52 part-00002-646db333-5c2c-4aa6-8d0c-fdaa759108df-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar 12 20:52 part-00003-646db333-5c2c-4aa6-8d0c-fdaa759108df-c000.snappy.parquet


As shown by the output above, each individual Parquet file is 26MB, which is closest to the given answer choice of 25MB.

###  Question 3

My solution (162,604) contains SQL, so please see README.

###  Question 4

My solution (90.6) contains SQL, so please see README.

###  Question 5

My solution (4040) does not require any code and was deduced by looking at the lecture materials.

###  Question 6

In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-12 21:21:10--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.171.57.103, 3.171.57.69, 3.171.57.179, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.171.57.103|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-12 21:21:10 (255 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [7]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

df_yellow.createOrReplaceTempView("yellow_data")
df_zones.createOrReplaceTempView("zones")

In [9]:
least_frequent_query = spark.sql("""
    SELECT 
        z.Zone AS pickup_zone_name, 
        COUNT(1) AS total_trips
    FROM 
        yellow_data y
    JOIN 
        zones z ON y.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY 
        total_trips ASC
""")

least_frequent_query.show(truncate=False)

[Stage 10:>                                                         (0 + 2) / 2]

+---------------------------------------------+-----------+
|pickup_zone_name                             |total_trips|
+---------------------------------------------+-----------+
|Governor's Island/Ellis Island/Liberty Island|1          |
|Eltingville/Annadale/Prince's Bay            |1          |
|Arden Heights                                |1          |
|Port Richmond                                |3          |
|Rikers Island                                |4          |
|Rossville/Woodrow                            |4          |
|Great Kills                                  |4          |
|Green-Wood Cemetery                          |4          |
|Jamaica Bay                                  |5          |
|Westerleigh                                  |12         |
|New Dorp/Midland Beach                       |14         |
|Oakwood                                      |14         |
|Crotona Park                                 |14         |
|West Brighton                          